<a href="https://colab.research.google.com/github/alansiny/INTERSHIP-AI-ML/blob/main/Movie_Review_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
df =pd.read_csv("/content/IMDB Dataset (1).csv",encoding='latin1',)
print(df.head())
print(df['sentiment'].value_counts())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [2]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [3]:
df['sentiment']=df['sentiment'].map({
    'positive':1,
    'negative':0
})

In [4]:
#preprocessing
import re
negation_words=[
    "not good","not bad","not great","don't like","didin't like","never liked", "wasn't good","isn't good","no good"

]
def clean_text(text):
  text=re.sub(r"[^a-zA-Z\s']","",text)
  for phrase in negation_words:
    text=text.replace(phrase,phrase.replace(" ","_"))
  return text

In [5]:
#splitting
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(df['review'],df['sentiment'],test_size=0.2,random_state=42)



In [6]:
#model training
from tensorflow.keras.preprocessing.text import Tokenizer
from  tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size=20000
max_len=250

tokenizer=Tokenizer(num_words=vocab_size,oov_token="<OOV>")#out of vocabulary
tokenizer.fit_on_texts(X_train)

X_train_seq=tokenizer.texts_to_sequences(X_train)
X_test_seq=tokenizer.texts_to_sequences(X_test) # Corrected: use X_test here

X_train_pad=pad_sequences(X_train_seq,maxlen=max_len,padding='post')
X_test_pad=pad_sequences(X_test_seq,maxlen=max_len,padding='post')

In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,Dropout

model=Sequential([
    Embedding(vocab_size,100),
    LSTM(128,dropout=0.3,recurrent_dropout=0.3, return_sequences=True),
    LSTM(64,dropout=0.3,recurrent_dropout=0.3),
    Dense(64,activation='relu'),
    Dropout(0.5),
    Dense(1,activation='sigmoid')
])

In [8]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']

)
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [9]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
history=model.fit(
    X_train_pad,y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 535s 1s/step - accuracy: 0.5648 - loss: 0.6654 - val_accuracy: 0.6610 - val_loss: 0.6185
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 541s 1s/step - accuracy: 0.7428 - loss: 0.5401 - val_accuracy: 0.7154 - val_loss: 0.5547
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 571s 1s/step - accuracy: 0.7880 - loss: 0.4888 - val_accuracy: 0.8125 - val_loss: 0.4448
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 514s 1s/step - accuracy: 0.8327 - loss: 0.4022 - val_accuracy: 0.8249 - val_loss: 0.4275
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 576s 1s/step - accuracy: 0.8522 - loss: 0.3565 - val_accuracy: 0.8451 - val_loss: 0.3905


In [10]:
loss,acc=model.evaluate(X_test_pad,y_test)
print("Test Accuracy:",acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 47s 147ms/step - accuracy: 0.8446 - loss: 0.3909
Test Accuracy: 0.8446000218391418


In [16]:
def predict_sentiment(review):
  review=clean_text(review)
  seq=tokenizer.texts_to_sequences([review])
  padded_sequence=pad_sequences(seq,maxlen=max_len,padding='post')
  prediction=model.predict(padded_sequence)[0][0]
  print("\nReview",review)
  print("Score:",prediction)

  if prediction>=0.5:
    print("Sentiment:Positive")
  else:
    print("Sentiment:Negative")


In [17]:
predict_sentiment("Movie is really good and amazing")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step

Review Movie is really good and amazing
Score: 0.71255517
Sentiment:Positive
